In [2]:
# ============================================================
# EXPERIMENT 1b
# EXPLORATORY DATA ANALYSIS AND DATA CLEANING
# CALIFORNIA HOUSING DATASET
# ============================================================

# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import pandas as pd
import numpy as np

from IPython.display import display, clear_output
import ipywidgets as widgets

print("Required libraries imported successfully.")


# ============================================================
# 2. LOAD THE DATASET
# ============================================================

path = "/housing prices.csv"

print("\nDataset path:", path)

df = pd.read_csv(path)

print("Dataset loaded successfully.")


# ============================================================
# 3. DISPLAY FIRST 5 AND LAST 5 RECORDS
# ============================================================

print("\nFirst 5 rows:")
display(df.head())

print("\nLast 5 rows:")
display(df.tail())


# ============================================================
# 4. DISPLAY DATASET INFORMATION
# ============================================================

print("\nDataset information:")
df.info()


# ============================================================
# 5. DISPLAY DATASET SHAPE
# ============================================================

print("\nShape:")
print(df.shape)


# ============================================================
# 6. DISPLAY COLUMN NAMES
# ============================================================

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# 7. DISPLAY DATA TYPES
# ============================================================

print("\nData types:")
print(df.dtypes)


# ============================================================
# 8. DESCRIPTIVE STATISTICAL INFORMATION
# ============================================================

print("\nStatistical description:")
display(df.describe())

print("\nCategorical column description:")
display(df.describe(include="object"))


# ============================================================
# 9. IDENTIFY MISSING VALUES
# ============================================================

print("\nMissing values:")
display(df.isnull().sum())


print("\nMissing value percentage:")
missing_percentage = (
    df.isnull().sum() / len(df) * 100
)

display(
    missing_percentage.to_frame(
        "Missing Percentage"
    )
)


# ============================================================
# 10. CHECK BLANK ROWS AND DUPLICATES
# ============================================================

print("\nNumber of completely blank rows:")
print(df.isnull().all(axis=1).sum())

print("\nNumber of duplicate records:")
print(df.duplicated().sum())


# ============================================================
# 11. REMOVE BLANK ROWS AND DUPLICATES
# ============================================================

df = df.dropna(how="all")
df = df.drop_duplicates()

print("\nDataset after removing blank rows and duplicates:")
print(df.shape)


# ============================================================
# 12. DATA CLEANING AND STANDARDIZATION
# ============================================================

# Standardize categorical attribute
if "ocean_proximity" in df.columns:

    df["ocean_proximity"] = (
        df["ocean_proximity"]
        .astype(str)
        .str.strip()
        .str.upper()
    )


# Numerical attributes
numeric_columns = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "median_house_value"
]


# Convert numerical attributes to numeric
for column in numeric_columns:

    if column in df.columns:

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )


print("\nData cleaning and standardization completed.")


# ============================================================
# 13. HANDLE MISSING VALUES
# ============================================================

for column in numeric_columns:

    if column in df.columns:

        if df[column].isnull().sum() > 0:

            df[column] = df[column].fillna(
                df[column].median()
            )


# Handle categorical missing values
categorical_columns = df.select_dtypes(
    include="object"
).columns

for column in categorical_columns:

    if df[column].isnull().sum() > 0:

        df[column] = df[column].fillna(
            df[column].mode()[0]
        )


print("\nMissing values handled successfully.")


# ============================================================
# 14. VERIFY CLEANED DATASET
# ============================================================

print("\nMissing values after cleaning:")
display(df.isnull().sum())

print("\nDuplicate records after cleaning:")
print(df.duplicated().sum())

print("\nCleaned dataset shape:")
print(df.shape)


# ============================================================
# 15. HOUSING-VALUE ANALYSIS
# ============================================================

print("\nHousing value statistics:")

housing_value_statistics = df[
    "median_house_value"
].agg([
    "count",
    "mean",
    "median",
    "std",
    "min",
    "max"
])

display(
    housing_value_statistics.to_frame(
        "Value"
    )
)


# ============================================================
# 16. OCEAN-PROXIMITY-WISE ANALYSIS
# ============================================================

print("\nOcean-proximity-wise housing analysis:")

ocean_analysis = (
    df.groupby("ocean_proximity")
    .agg(
        Number_of_Houses=(
            "median_house_value",
            "count"
        ),
        Average_House_Value=(
            "median_house_value",
            "mean"
        ),
        Average_Income=(
            "median_income",
            "mean"
        )
    )
    .sort_values(
        "Average_House_Value",
        ascending=False
    )
)

display(ocean_analysis)


# ============================================================
# 17. HOUSING FEATURE-WISE ANALYSIS
# ============================================================

print("\nHousing feature-wise statistics:")

feature_analysis = df[
    [
        "total_rooms",
        "total_bedrooms",
        "population",
        "households",
        "median_income",
        "median_house_value"
    ]
].describe().T

display(feature_analysis)


# ============================================================
# 18. CORRELATION ANALYSIS
# ============================================================

print("\nCorrelation:")

correlation = df.corr(
    numeric_only=True
)

display(correlation)


# ============================================================
# 19. TOP 10 HOUSES BY HOUSE VALUE
# ============================================================

print("\nTop 10 houses by median house value:")

top_10_houses = (
    df.sort_values(
        "median_house_value",
        ascending=False
    )
    .head(10)
)

display(
    top_10_houses[
        [
            "longitude",
            "latitude",
            "median_income",
            "total_rooms",
            "median_house_value",
            "ocean_proximity"
        ]
    ]
)


# ============================================================
# 20. TOP 10 RECORDS BY MEDIAN INCOME
# ============================================================

print("\nTop 10 records by median income:")

top_10_income = (
    df.sort_values(
        "median_income",
        ascending=False
    )
    .head(10)
)

display(
    top_10_income[
        [
            "longitude",
            "latitude",
            "median_income",
            "median_house_value",
            "ocean_proximity"
        ]
    ]
)


# ============================================================
# 21. MINIMUM HOUSE VALUE PARAMETER
# ============================================================

print("\nMinimum House Value Parameter:")

min_house_value = widgets.IntSlider(
    value=100000,
    min=int(df["median_house_value"].min()),
    max=int(df["median_house_value"].max()),
    step=10000,
    description="Minimum Value:",
    continuous_update=False
)

parameter_output = widgets.Output()


def minimum_value_analysis(change=None):

    with parameter_output:

        clear_output(wait=True)

        selected_value = min_house_value.value

        filtered_df = df[
            df["median_house_value"]
            >= selected_value
        ]

        print(
            "Minimum House Value:",
            f"${selected_value:,.0f}"
        )

        print(
            "Number of matching records:",
            len(filtered_df)
        )

        print("\nAverage House Value:")

        if len(filtered_df) > 0:

            print(
                f"${filtered_df['median_house_value'].mean():,.2f}"
            )

            print("\nAverage Median Income:")

            print(
                f"{filtered_df['median_income'].mean():.2f}"
            )

            print("\nOcean Proximity Distribution:")

            display(
                filtered_df[
                    "ocean_proximity"
                ]
                .value_counts()
                .to_frame(
                    "Number of Houses"
                )
            )

        else:

            print(
                "No records found."
            )


min_house_value.observe(
    minimum_value_analysis,
    names="value"
)

display(
    min_house_value,
    parameter_output
)

minimum_value_analysis()


# ============================================================
# 22. INTERACTIVE OCEAN PROXIMITY FILTER
# ============================================================

print("\nInteractive Ocean Proximity Filter:")

ocean_options = [
    "ALL"
] + sorted(
    df["ocean_proximity"].unique().tolist()
)

ocean_filter = widgets.Dropdown(
    options=ocean_options,
    value="ALL",
    description="Location:"
)

filter_output = widgets.Output()


def filter_by_ocean(change=None):

    with filter_output:

        clear_output(wait=True)

        selected_location = ocean_filter.value

        if selected_location == "ALL":

            filtered_df = df.copy()

        else:

            filtered_df = df[
                df["ocean_proximity"]
                == selected_location
            ]

        print(
            "Selected Location:",
            selected_location
        )

        print(
            "Number of Records:",
            len(filtered_df)
        )

        print("\nFiltered Dataset:")

        display(
            filtered_df.head(10)
        )


ocean_filter.observe(
    filter_by_ocean,
    names="value"
)

display(
    ocean_filter,
    filter_output
)

filter_by_ocean()


# ============================================================
# 23. INTERACTIVE HOUSING ANALYSIS DASHBOARD
# ============================================================

print("\nInteractive Housing Analysis Dashboard:")

dashboard_location = widgets.Dropdown(
    options=ocean_options,
    value="ALL",
    description="Ocean Proximity:"
)

dashboard_value = widgets.IntSlider(
    value=100000,
    min=int(df["median_house_value"].min()),
    max=int(df["median_house_value"].max()),
    step=10000,
    description="Minimum Value:",
    continuous_update=False
)

dashboard_output = widgets.Output()


def update_dashboard(change=None):

    with dashboard_output:

        clear_output(wait=True)

        selected_location = (
            dashboard_location.value
        )

        selected_value = (
            dashboard_value.value
        )

        if selected_location == "ALL":

            filtered_df = df.copy()

        else:

            filtered_df = df[
                df["ocean_proximity"]
                == selected_location
            ]

        filtered_df = filtered_df[
            filtered_df["median_house_value"]
            >= selected_value
        ]

        print(
            "=========================================="
        )

        print(
            "CALIFORNIA HOUSING ANALYSIS DASHBOARD"
        )

        print(
            "=========================================="
        )

        print(
            "Ocean Proximity:",
            selected_location
        )

        print(
            "Minimum House Value:",
            f"${selected_value:,.0f}"
        )

        print(
            "Number of Records:",
            len(filtered_df)
        )

        if len(filtered_df) > 0:

            print(
                "\nAverage House Value:",
                f"${filtered_df['median_house_value'].mean():,.2f}"
            )

            print(
                "Average Median Income:",
                f"{filtered_df['median_income'].mean():.2f}"
            )

            print(
                "Average Total Rooms:",
                f"{filtered_df['total_rooms'].mean():,.2f}"
            )

            print("\nFiltered Records:")

            display(
                filtered_df.head(10)
            )

        else:

            print(
                "\nNo records found for the selected conditions."
            )


dashboard_location.observe(
    update_dashboard,
    names="value"
)

dashboard_value.observe(
    update_dashboard,
    names="value"
)

display(
    widgets.VBox([
        dashboard_location,
        dashboard_value
    ])
)

display(
    dashboard_output
)

update_dashboard()


# ============================================================
# 24. EXPORT CLEANED DATASET
# ============================================================

output_file = "/content/housing_prices_cleaned.csv"

df.to_csv(
    output_file,
    index=False
)

print("\nCSV file created successfully.")

print(
    "Cleaned dataset saved as:",
    output_file
)


# ============================================================
# 25. FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)

print(
    "Exploratory Data Analysis and Data Cleaning "
    "were successfully performed on the California "
    "Housing Dataset."
)

print(
    "The dataset structure, dimensions, data types, "
    "statistical characteristics and missing values "
    "were analyzed."
)

print(
    "Blank rows and duplicate records were removed "
    "and the required data cleaning and standardization "
    "were performed."
)

print(
    "Housing-value, ocean-proximity-wise and "
    "feature-wise analyses were performed."
)

print(
    "Interactive filtering, minimum house value "
    "parameter analysis and an interactive housing "
    "analysis dashboard were created."
)

print(
    "The cleaned dataset was exported successfully "
    "in CSV format."
)

print("\nExperiment completed successfully.")

Required libraries imported successfully.

Dataset path: /housing prices.csv
Dataset loaded successfully.

First 5 rows:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY



Last 5 rows:


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND
20639,-121.24,39.37,16.0,2785.0,616.0,1387.0,530.0,2.3886,89400.0,INLAND



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  object 
dtypes: float64(9), object(1)
memory usage: 1.6+ MB

Shape:
(20640, 10)

Columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']

Data types:
longitude          

...

Dropdown(description='Location:', options=('ALL', '<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'), …

Output()


Interactive Housing Analysis Dashboard:


Output()


CSV file created successfully.
Cleaned dataset saved as: /content/housing_prices_cleaned.csv

RESULT
Exploratory Data Analysis and Data Cleaning were successfully performed on the California Housing Dataset.
The dataset structure, dimensions, data types, statistical characteristics and missing values were analyzed.
Blank rows and duplicate records were removed and the required data cleaning and standardization were performed.
Housing-value, ocean-proximity-wise and feature-wise analyses were performed.
Interactive filtering, minimum house value parameter analysis and an interactive housing analysis dashboard were created.
The cleaned dataset was exported successfully in CSV format.

Experiment completed successfully.
